In [649]:
!pip install ipywidgets
import ipywidgets as widgets
from ipywidgets import interact, interactive


[notice] A new release of pip is available: 23.3.2 -> 26.1.1
[notice] To update, run: C:\Users\mbieg\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe -m pip install --upgrade pip


In [650]:
import numpy as np
import matplotlib.pyplot as plt

def resolve_velocity(speed, course_deg):
    theta = np.radians(course_deg)
    return np.array([speed * np.cos(theta), speed * np.sin(theta)])

def bearing_and_range(R):
    x, y = R
    bearing = (np.degrees(np.arctan2(y, x)) + 360) % 360
    rng = np.hypot(x, y)
    return bearing, rng

In [651]:
import numpy as np

def resolve_velocity(speed, course_deg):
    """
    speed: knots (or any units, as long as you're consistent)
    course_deg: 0° = +x, 90° = +y (submarine-style)
    """
    theta = np.radians(course_deg)
    return np.array([speed * np.cos(theta), speed * np.sin(theta)])

class PositionKeeper:
    def __init__(self, R0, own_course, own_speed, tgt_course, tgt_speed):
        self.R = np.array(R0, dtype=float)  # relative position (x,y) target wrt own ship
        self.own_course = own_course
        self.own_speed = own_speed
        self.tgt_course = tgt_course
        self.tgt_speed = tgt_speed

    def step(self, dt):
        V_own = resolve_velocity(self.own_speed, self.own_course)
        V_tgt = resolve_velocity(self.tgt_speed, self.tgt_course)
        V_rel = V_tgt - V_own
        self.R = self.R + V_rel * dt
        return self.R, V_rel

In [652]:
def bearing_and_range(R):
    x, y = R
    bearing = (np.degrees(np.arctan2(y, x)) + 360) % 360
    rng = np.hypot(x, y)
    return bearing, rng

In [653]:
class PositionKeeper:
    def __init__(self, R0, own_course, own_speed, tgt_course, tgt_speed):
        self.R = np.array(R0, dtype=float)
        self.own_course = own_course
        self.own_speed = own_speed
        self.tgt_course = tgt_course
        self.tgt_speed = tgt_speed

    def step(self, dt):
        V_own = resolve_velocity(self.own_speed, self.own_course)
        V_tgt = resolve_velocity(self.tgt_speed, self.tgt_course)
        V_rel = V_tgt - V_own
        self.R = self.R + V_rel * dt
        bearing, rng = bearing_and_range(self.R)
        return self.R, V_rel, bearing, rng

In [654]:
def compute_gyro_angle(R, V_rel, V_torp):
    x, y = R
    vx, vy = V_rel

    a = np.dot(V_rel, V_rel) - V_torp**2
    b = 2 * np.dot(R, V_rel)
    c = np.dot(R, R)

    disc = b*b - 4*a*c
    if disc < 0:
        return np.nan

    t1 = (-b + np.sqrt(disc)) / (2*a)
    t2 = (-b - np.sqrt(disc)) / (2*a)
    t_candidates = [t for t in (t1, t2) if t > 0]

    if not t_candidates:
        return np.nan

    t = min(t_candidates)
    intercept_point = R + V_rel * t
    angle = (np.degrees(np.arctan2(intercept_point[1], intercept_point[0])) + 360) % 360
    return angle


In [655]:
# Compute intercept point for plotting
# (same t used inside compute_gyro_angle)
def intercept_point(R, V_rel, V_torp):
    x, y = R
    vx, vy = V_rel

    a = np.dot(V_rel, V_rel) - V_torp**2
    b = 2 * np.dot(R, V_rel)
    c = np.dot(R, R)

    disc = b*b - 4*a*c
    if disc < 0:
        return np.array([np.nan, np.nan])

    t1 = (-b + np.sqrt(disc)) / (2*a)
    t2 = (-b - np.sqrt(disc)) / (2*a)
    t_candidates = [t for t in (t1, t2) if t > 0]

    if not t_candidates:
        return np.array([np.nan, np.nan])

    t = min(t_candidates)
    return R + V_rel * t


In [656]:
# True scenario
true_own_course = 0.0
true_own_speed  = 5.0
true_tgt_course = 90.0
true_tgt_speed  = 12.0
true_R0         = np.array([3000.0, -500.0])

# TDC guesses (intentionally wrong)
guess_AOB_error = -5.0
tgt_course_guess = true_tgt_course + guess_AOB_error
tgt_speed_guess  = 10.0
R0_guess         = np.array([3000.0, -500.0])

# Torpedo
V_torp = 46.0

# Instantiate BOTH integrators
true_pk = PositionKeeper(
    R0=true_R0,
    own_course=true_own_course,
    own_speed=true_own_speed,
    tgt_course=true_tgt_course,
    tgt_speed=true_tgt_speed
)

tdc_pk = PositionKeeper(
    R0=R0_guess,
    own_course=true_own_course,
    own_speed=true_own_speed,
    tgt_course=tgt_course_guess,
    tgt_speed=tgt_speed_guess
)



In [657]:
def run_tdc_sim(AOB_guess, speed_guess, range_guess):

    # --- TRUE TARGET PK ---
    true_pk = PositionKeeper(
        R0=true_R0,
        own_course=true_own_course,
        own_speed=true_own_speed,
        tgt_course=true_tgt_course,
        tgt_speed=true_tgt_speed
    )

    # --- SYNTHETIC TARGET PK ---
    if abs(AOB_guess) < 1e-6:
        tgt_course_guess = true_tgt_course
    else:
        tgt_course_guess = true_tgt_course + AOB_guess

    if abs(speed_guess - true_tgt_speed) < 1e-6:
        speed_guess = true_tgt_speed

    R0_guess = true_R0.copy()

    tdc_pk = PositionKeeper(
        R0=R0_guess,
        own_course=true_own_course,
        own_speed=true_own_speed,
        tgt_course=tgt_course_guess,
        tgt_speed=speed_guess
    )

    # ============================================================
    # === COMPUTE TIME-TO-IMPACT (t_hit) FROM INITIAL GEOMETRY ===
    # ============================================================

    R0 = R0_guess.copy()

    # Own ship velocity
    V_own = np.array([
        true_own_speed * np.sin(np.deg2rad(true_own_course)),
        true_own_speed * np.cos(np.deg2rad(true_own_course))
    ])

    # Target velocity (synthetic)
    V_tgt = np.array([
        speed_guess * np.sin(np.deg2rad(tgt_course_guess)),
        speed_guess * np.cos(np.deg2rad(tgt_course_guess))
    ])

    # Relative velocity
    V_rel0 = V_tgt - V_own

    # Quadratic coefficients for |R0 + V_rel t| = V_torp t
    a = np.dot(V_rel0, V_rel0) - V_torp**2
    b = 2.0 * np.dot(R0, V_rel0)
    c = np.dot(R0, R0)

    disc = b*b - 4*a*c

    if disc < 0:
        t_hit = 30.0
    else:
        sqrt_disc = np.sqrt(disc)
        t1 = (-b + sqrt_disc) / (2*a)
        t2 = (-b - sqrt_disc) / (2*a)
        candidates = [t for t in (t1, t2) if t > 0]
        t_hit = min(candidates) if candidates else 30.0

    # --- SIMULATION TIME ---
    dt = 0.5
    T  = t_hit
    steps = int(T / dt)

    # --- STORAGE ---
    times = []
    bearing_true_list = []
    bearing_syn_list  = []
    bearing_residuals = []
    bearing_rate_true_list = []
    bearing_rate_syn_list  = []
    gyro_list = []
    true_positions = []
    syn_positions = []

    # --- SIMULATION LOOP (steps + 1 ensures final time is included) ---
    for i in range(steps + 1):
        t = i * dt

        R_true, V_rel_true, bearing_true, range_true = true_pk.step(dt)
        R_syn,  V_rel_syn,  bearing_syn,  range_syn  = tdc_pk.step(dt)

        if i == 0:
            br_true = 0
            br_syn  = 0
        else:
            br_true = (bearing_true - bearing_true_prev) / dt
            br_syn  = (bearing_syn  - bearing_syn_prev)  / dt

        bearing_true_prev = bearing_true
        bearing_syn_prev  = bearing_syn

        gyro = compute_gyro_angle(R_syn, V_rel_syn, V_torp)

        times.append(t)
        bearing_true_list.append(bearing_true)
        bearing_syn_list.append(bearing_syn)
        bearing_residuals.append((bearing_true - bearing_syn + 540) % 360 - 180)
        bearing_rate_true_list.append(br_true)
        bearing_rate_syn_list.append(br_syn)
        gyro_list.append(gyro)
        true_positions.append(R_true.copy())
        syn_positions.append(R_syn.copy())

    # ============================================================
    # === FINAL INTERCEPT POINT (Option A) ========================
    # ============================================================

    I_final = intercept_point(R_syn, V_rel_syn, V_torp)

    # ============================================================
    # === PLOTTING ===============================================
    # ============================================================

    plt.close('all')
    fig, axs = plt.subplots(4, 1, figsize=(8, 12))

    # 1. Bearings
    axs[0].plot(times, bearing_true_list, label='True Bearing')
    axs[0].plot(times, bearing_syn_list,  label='Synthetic Bearing')
    axs[0].legend()
    axs[0].grid(True)

    # 2. Residual
    axs[1].plot(times, bearing_residuals, label='Residual')
    axs[1].legend()
    axs[1].grid(True)

    # 3. Bearing rate
    axs[2].plot(times, bearing_rate_true_list, label='True Rate')
    axs[2].plot(times, bearing_rate_syn_list,  label='Synthetic Rate')
    axs[2].legend()
    axs[2].grid(True)

    # 4. Geometry
    axs[3].plot(
        [p[0] for p in true_positions],
        [p[1] for p in true_positions],
        label='True Target Track',
        linewidth=3
    )

    axs[3].plot(
        [p[0] for p in syn_positions],
        [p[1] for p in syn_positions],
        label='Synthetic Target Track',
        linewidth=1
    )

    axs[3].scatter(0, 0, color='black', marker='x', label='Own Ship')

    # Torpedo run to FINAL intercept point
    if not np.isnan(I_final[0]):
        axs[3].plot([0, I_final[0]], [0, I_final[1]], 'k--', label='Torpedo Run')

    # Auto-zoom
    xs = [p[0] for p in true_positions] + [p[0] for p in syn_positions] + [0, I_final[0]]
    ys = [p[1] for p in true_positions] + [p[1] for p in syn_positions] + [0, I_final[1]]

    pad = 600
    axs[3].set_xlim(min(xs) - pad, max(xs) + pad)
    axs[3].set_ylim(min(ys) - pad, max(ys) + pad)

    axs[3].set_aspect('equal', 'box')
    axs[3].legend()
    axs[3].grid(True)

    plt.tight_layout()
    return fig


In [658]:
AOB_slider = widgets.FloatSlider(min=-40, max=40, step=1, value=0)
speed_slider = widgets.FloatSlider(min=5, max=20, step=0.5, value=12)
range_slider = widgets.FloatSlider(min=1000, max=5000, step=100, value=3000)

ui = widgets.VBox([AOB_slider, speed_slider, range_slider])

out = widgets.interactive_output(
    run_tdc_sim,
    {
        'AOB_guess': AOB_slider,
        'speed_guess': speed_slider,
        'range_guess': range_slider
    }
)
display(ui,out)

Output()